# Agentes con Bases de Datos — Text-to-SQL

## Módulo 3 — Tool Use | Semana 7

---

## Objetivos de Aprendizaje

| # | Objetivo |
|---|----------|
| 1 | Entender el patrón **Text-to-SQL** y sus problemas principales (schema, errores, seguridad) |
| 2 | Diseñar un set de **tools especializadas para databases** (list_tables, get_schema, query) |
| 3 | Integrar un agente con **PostgreSQL** usando `asyncpg` y connection pooling |
| 4 | Generar **queries geoespaciales con PostGIS** (distancias, proximidad) |
| 5 | Implementar un **pipeline de validación de queries** seguro para producción |

---

> **Contexto**: Seguimos trabajando con **Klimbook** (app de escalada). En esta lección el agente va a consultar la base de datos PostgreSQL de Klimbook para responder preguntas sobre usuarios, rutas, ascensos y crags.

---

## 7.1 Text-to-SQL con LLMs

La idea es simple: el usuario pregunta en lenguaje natural, el agente genera un query SQL, lo ejecuta contra tu base de datos, e interpreta los resultados.

```
Usuario: "Cuantos usuarios se registraron este mes?"
    |
    v
Claude: genera SQL
    SELECT COUNT(*) FROM users 
    WHERE created_at >= '2026-04-01'
    |
    v
Tu codigo: ejecuta el SQL en PostgreSQL
    -> {count: 47}
    |
    v
Claude: interpreta el resultado
    "Se registraron 47 usuarios este mes."
```

Suena simple pero hay **tres problemas** que resolver antes de que esto funcione en producción.

### Problema 1: Claude necesita conocer tu schema

Claude no sabe qué tablas tienes, qué columnas tienen, ni qué tipos de datos usan. Si no le das esa información, va a **inventar** nombres de tablas y columnas (hallucination).

Hay dos formas de darle el schema:

| Forma | Ventaja | Desventaja |
|-------|---------|------------|
| **Hardcodear en el system prompt** | Simple, rápido | Si cambias el schema, tienes que actualizar el prompt manualmente |
| **Que el agente lo consulte con tools** | Se adapta automáticamente a cambios | Requiere tool calls extra (más tokens, más lento) |

La **forma 2 es mejor** para producción porque si agregas columnas o creas tablas, el agente se adapta sin modificar el código.

In [ ]:
# Forma 1: Hardcodear el schema en el system prompt
SYSTEM_HARDCODED = """Tienes acceso a una base de datos PostgreSQL con este schema:

CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(50) NOT NULL,
    email VARCHAR(100) NOT NULL,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE routes (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    grade VARCHAR(10),
    type VARCHAR(20),  -- 'sport', 'boulder', 'trad'
    crag_id INTEGER REFERENCES crags(id),
    geom GEOMETRY(Point, 4326)
);
"""

# Forma 2: Que el agente consulte el schema con tools (MEJOR)
# Lo implementaremos en la seccion 7.2 con:
#   - list_tables()        -> que tablas existen
#   - get_table_schema()   -> columnas, tipos, constraints de una tabla
#   - get_sample_data()    -> rows de ejemplo para entender los datos

### Problema 2: Claude puede generar SQL incorrecto

Estos son los errores más comunes que Claude comete al generar SQL:

| Error | Ejemplo | Causa |
|-------|---------|-------|
| **Columnas inventadas** | `SELECT user_name FROM users` (la columna es `username`) | Hallucination — Claude adivina el nombre |
| **JOINs incorrectos** | `JOIN routes ON users.id = routes.id` (debería ser `routes.user_id`) | No entiende las relaciones entre tablas |
| **Dialecto equivocado** | `DATEADD(month, -1, GETDATE())` — eso es SQL Server | Claude mezcla dialectos si no le especificas |
| **Queries sin LIMIT** | `SELECT * FROM ascents` (500,000 rows) | Puede matar tu servidor si la tabla es grande |

Por eso es crucial darle a Claude el **schema exacto** y especificar que el dialecto es **PostgreSQL**.

In [ ]:
# Ejemplos de SQL incorrecto que Claude podria generar:

# Error 1: Columnas que no existen
bad_sql_1 = "SELECT user_name FROM users;"
# La columna se llama 'username', no 'user_name'

# Error 2: JOINs incorrectos
bad_sql_2 = "SELECT * FROM users JOIN routes ON users.id = routes.id;"
# El JOIN deberia ser ON users.id = routes.user_id (si existiera)

# Error 3: Funciones de otro dialecto SQL
bad_sql_3 = "SELECT DATEADD(month, -1, GETDATE());"
# Eso es SQL Server. En PostgreSQL es:
good_sql_3 = "SELECT NOW() - INTERVAL '1 month';"

# Error 4: Queries que retornan demasiadas rows
bad_sql_4 = "SELECT * FROM ascents;"
# Si tienes 500,000 ascensos, esto mata tu servidor
good_sql_4 = "SELECT * FROM ascents LIMIT 100;"

### Problema 3 (el más importante): SEGURIDAD

Este es el punto más crítico de todo el módulo. Si Claude genera SQL y tu código lo ejecuta, **estás dándole a un LLM la capacidad de modificar tu base de datos**. Un prompt injection podría hacer que Claude genere:

```sql
DROP TABLE users;
DELETE FROM ascents WHERE 1=1;
UPDATE users SET role = 'admin' WHERE email = 'attacker@evil.com';
```

Las protecciones son **capas de defensa** — cada una es insuficiente sola, pero juntas hacen el sistema seguro:

| Capa | Protección | Qué previene |
|------|-----------|--------------|
| 1 | **Solo SELECT** — parsear SQL y rechazar todo lo demás | DROP, DELETE, INSERT, UPDATE |
| 2 | **Usuario read-only** — permisos de PostgreSQL | Incluso si el parseo falla, la DB rechaza writes |
| 3 | **LIMIT obligatorio** — agregar si no existe | Queries que retornan millones de rows |
| 4 | **Timeout** — cancelar queries lentos | JOINs cartesianos, full table scans |

> **Defensa en profundidad**: nunca confíes en una sola capa. La capa 1 (parseo) puede tener bugs. La capa 2 (permisos de DB) es tu red de seguridad si el parseo falla.

In [ ]:
import sqlparse


def is_safe_query(sql: str) -> bool:
    """
    Verifica que el SQL sea un SELECT y nada mas.
    Retorna True solo si es seguro ejecutar.
    
    Usa sqlparse para parsear el SQL y verificar que:
    1. Es un statement valido
    2. El tipo es SELECT (no INSERT, UPDATE, DELETE, etc.)
    3. No contiene keywords peligrosos ni en subconsultas
    """
    parsed = sqlparse.parse(sql.strip())

    if not parsed:
        return False

    for statement in parsed:
        # sqlparse detecta el tipo de statement: SELECT, INSERT, etc.
        stmt_type = statement.get_type()

        if stmt_type != "SELECT":
            return False

        # Buscar keywords peligrosos incluso dentro de subconsultas.
        # Un atacante podria esconder un DELETE dentro de un SELECT:
        #   SELECT * FROM (DELETE FROM users RETURNING *) sub;
        sql_upper = sql.upper()
        dangerous_keywords = [
            "INSERT", "UPDATE", "DELETE", "DROP", "ALTER",
            "CREATE", "TRUNCATE", "GRANT", "REVOKE",
            "EXEC", "EXECUTE", "INTO",  # SELECT INTO crea tablas
        ]
        for keyword in dangerous_keywords:
            # Buscar la palabra COMPLETA, no como substring.
            # Sin esto, bloqueariamos "UPDATED_AT" o "SELECTED".
            if f" {keyword} " in f" {sql_upper} ":
                return False

    return True


def add_limit(sql: str, max_rows: int = 100) -> str:
    """
    Agrega LIMIT al query si no lo tiene.
    Protege contra queries que retornan millones de rows.
    """
    sql_upper = sql.upper().strip()
    if "LIMIT" not in sql_upper:
        sql = sql.rstrip(";").strip()
        sql = f"{sql} LIMIT {max_rows}"
    return sql


# Probar las protecciones
print("=== Pruebas de is_safe_query ===")
test_queries = [
    ("SELECT * FROM users", True),
    ("DELETE FROM users", False),
    ("DROP TABLE users", False),
    ("SELECT * FROM (DELETE FROM users RETURNING *) sub", False),
    ("SELECT updated_at FROM users", True),  # 'updated_at' NO es 'UPDATE'
    ("SELECT INTO new_table FROM users", False),  # SELECT INTO crea tablas
]

for sql, expected in test_queries:
    result = is_safe_query(sql)
    status = "OK" if result == expected else "FAIL"
    print(f"  [{status}] is_safe_query('{sql[:50]}') = {result}")

print("\n=== Pruebas de add_limit ===")
print(f"  Con LIMIT:    '{add_limit('SELECT * FROM users LIMIT 10')}'")
print(f"  Sin LIMIT:    '{add_limit('SELECT * FROM users')}'")
print(f"  Con ; final:  '{add_limit('SELECT * FROM users;')}'")


In [ ]:
# Proteccion a nivel de PostgreSQL: crear un usuario read-only.
# Esto es tu RED DE SEGURIDAD -- incluso si is_safe_query() tiene bugs,
# PostgreSQL rechaza cualquier write.

# Ejecutar esto en psql como superuser:
SETUP_READONLY_USER = """
-- Crear usuario con password seguro
CREATE USER readonly_agent WITH PASSWORD 'secure_password';

-- Permitir conectarse a la base de datos
GRANT CONNECT ON DATABASE klimbook TO readonly_agent;

-- Permitir usar el schema public
GRANT USAGE ON SCHEMA public TO readonly_agent;

-- Permitir SELECT en todas las tablas (presentes y futuras)
GRANT SELECT ON ALL TABLES IN SCHEMA public TO readonly_agent;
ALTER DEFAULT PRIVILEGES IN SCHEMA public 
    GRANT SELECT ON TABLES TO readonly_agent;
"""

print(SETUP_READONLY_USER)

---

## 7.2 Diseño de Tools para Databases

En vez de darle a Claude una sola tool "ejecuta este SQL", le damos **varias tools especializadas** que controlan el acceso de forma granular. Esto es mucho más seguro y produce mejores resultados porque Claude **entiende el schema antes de escribir SQL**.

### El set de tools recomendado

| Tool | Propósito | Cuándo la usa Claude |
|------|----------|---------------------|
| `list_tables()` | Retorna los nombres de todas las tablas | **Primero** — para saber qué tablas existen |
| `get_table_schema(table)` | Columnas, tipos, constraints de UNA tabla | Para entender la estructura antes de hacer queries |
| `get_sample_data(table, n)` | N rows de ejemplo de una tabla | Para entender los datos reales (valores, formatos) |
| `query_database(sql)` | Ejecuta un SELECT con todas las protecciones | La única que ejecuta SQL — siempre al final |

### El flujo típico del agente

```
Usuario: "Cual es la ruta mas repetida?"
    |
Step 1: Claude llama list_tables()
    -> ["users", "routes", "ascents", "crags"]
    Claude piensa: "Necesito routes y ascents"
    |
Step 2: Claude llama get_table_schema("ascents")
    -> {columns: [id, route_id, user_id, style, ...]}
    Claude piensa: "ascents tiene route_id, puedo hacer JOIN con routes"
    |
Step 3: Claude llama get_table_schema("routes")
    -> {columns: [id, name, grade, ...]}
    Claude piensa: "Ahora se que columnas usar"
    |
Step 4: Claude llama query_database(sql=
    "SELECT r.name, r.grade, COUNT(a.id) as total_ascents
     FROM routes r JOIN ascents a ON r.id = a.route_id
     GROUP BY r.id, r.name, r.grade
     ORDER BY total_ascents DESC LIMIT 5")
    |
Step 5: Claude responde
    "La ruta mas repetida es La Esfinge (5.11a) con 47 ascensos."
```

Fíjate que Claude necesitó **4 tool calls** antes de hacer el query real. Parece ineficiente, pero es más seguro y produce mejores queries porque Claude entiende el schema antes de escribir SQL.

### Implementación de las tools

Las primeras dos tools (`list_tables` y `get_table_schema`) consultan `information_schema` — una "meta-tabla" de PostgreSQL que contiene información sobre la estructura de la base de datos. No ejecutan SQL del usuario, así que son intrínsecamente seguras.

In [ ]:
import asyncpg
import logging
import json

logger = logging.getLogger(__name__)

# Pool de conexiones -- se crea una vez y se reutiliza.
# Un pool mantiene varias conexiones abiertas a la DB y las
# reparte entre las requests. Mucho mas eficiente que
# abrir/cerrar una conexion por cada query.
pool: asyncpg.Pool = None


async def init_pool():
    """Crea el pool de conexiones a PostgreSQL."""
    global pool
    pool = await asyncpg.create_pool(
        host="localhost",
        port=5432,
        database="klimbook",
        user="readonly_agent",     # usuario read-only (capa de seguridad)
        password="secure_password",
        min_size=2,                # minimo 2 conexiones abiertas
        max_size=10,               # maximo 10 conexiones
        command_timeout=10,        # timeout de 10 segundos por query
    )
    logger.info("[DB] Pool creado")

### Tool 1: `list_tables()` — Qué tablas existen

In [ ]:
async def list_tables() -> dict:
    """
    Lista todas las tablas de la base de datos.
    
    Consulta information_schema, que es una "meta-tabla" de PostgreSQL
    que contiene informacion sobre la estructura de la DB.
    Esta query es segura porque solo lee metadata, no datos del usuario.
    """
    query = """
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public' 
        AND table_type = 'BASE TABLE'
        ORDER BY table_name
    """
    async with pool.acquire() as conn:
        rows = await conn.fetch(query)

    tables = [row["table_name"] for row in rows]
    return {"tables": tables, "count": len(tables)}

### Tool 2: `get_table_schema()` — Estructura de una tabla

Esta tool es la más importante para la calidad de los queries. Le dice a Claude exactamente qué columnas tiene cada tabla, sus tipos, y las **foreign keys** (que le permiten saber cómo hacer JOINs correctos).

In [ ]:
async def get_table_schema(table_name: str) -> dict:
    """
    Retorna el schema de una tabla: columnas, tipos, y constraints.
    
    Incluye primary keys y foreign keys para que Claude entienda
    las relaciones entre tablas y pueda hacer JOINs correctos.
    """
    # VALIDACION: Solo caracteres alfanumericos y underscore.
    # Esto previene SQL injection en el nombre de tabla.
    # Sin esto, alguien podria enviar: table_name = "users; DROP TABLE users"
    if not table_name.replace("_", "").isalnum():
        return {"error": f"Nombre de tabla invalido: {table_name}"}

    # Obtener columnas desde information_schema
    columns_query = """
        SELECT 
            column_name, 
            data_type, 
            is_nullable,
            column_default,
            character_maximum_length
        FROM information_schema.columns 
        WHERE table_schema = 'public' 
        AND table_name = $1
        ORDER BY ordinal_position
    """

    # Obtener primary keys
    pk_query = """
        SELECT kcu.column_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu 
            ON tc.constraint_name = kcu.constraint_name
        WHERE tc.table_name = $1 
        AND tc.constraint_type = 'PRIMARY KEY'
    """

    # Obtener foreign keys.
    # Las FK son importantes: le dicen a Claude como hacer JOINs.
    # Si ascents tiene FK a routes, Claude sabe que puede hacer:
    #   JOIN ascents ON routes.id = ascents.route_id
    fk_query = """
        SELECT
            kcu.column_name,
            ccu.table_name AS foreign_table,
            ccu.column_name AS foreign_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu 
            ON tc.constraint_name = kcu.constraint_name
        JOIN information_schema.constraint_column_usage ccu 
            ON tc.constraint_name = ccu.constraint_name
        WHERE tc.table_name = $1 
        AND tc.constraint_type = 'FOREIGN KEY'
    """

    async with pool.acquire() as conn:
        columns = await conn.fetch(columns_query, table_name)
        pks = await conn.fetch(pk_query, table_name)
        fks = await conn.fetch(fk_query, table_name)

    if not columns:
        return {"error": f"Tabla no encontrada: {table_name}"}

    pk_names = [row["column_name"] for row in pks]

    return {
        "table_name": table_name,
        "columns": [
            {
                "name": col["column_name"],
                "type": col["data_type"],
                "nullable": col["is_nullable"] == "YES",
                "default": col["column_default"],
                "max_length": col["character_maximum_length"],
                "is_primary_key": col["column_name"] in pk_names,
            }
            for col in columns
        ],
        "primary_keys": pk_names,
        "foreign_keys": [
            {
                "column": fk["column_name"],
                "references_table": fk["foreign_table"],
                "references_column": fk["foreign_column"],
            }
            for fk in fks
        ],
    }

### Tool 3: `get_sample_data()` — Ver datos reales

Esta tool es útil porque Claude puede ver los datos reales y entender el **formato**. Por ejemplo, si la columna `grade` tiene valores como `'5.10a'`, `'5.11b'`, `'V3'`, Claude sabe qué formato usar en sus queries en vez de adivinar.

In [ ]:
async def get_sample_data(table_name: str, n: int = 5) -> dict:
    """
    Retorna N rows de ejemplo de una tabla.
    Claude la usa para entender los datos reales (formatos, valores).
    """
    # Misma validacion de nombre de tabla
    if not table_name.replace("_", "").isalnum():
        return {"error": f"Nombre de tabla invalido: {table_name}"}

    # Limitar N para seguridad (minimo 1, maximo 20)
    n = min(max(1, n), 20)

    # El nombre de tabla va directo en el string porque asyncpg no permite
    # parametrizar nombres de tablas con $1. Por eso la validacion de arriba
    # es critica.
    query = f"SELECT * FROM {table_name} LIMIT {n}"

    async with pool.acquire() as conn:
        rows = await conn.fetch(query)

    if not rows:
        return {"table": table_name, "sample": [], "count": 0}

    # Convertir asyncpg Records a dicts serializables.
    # asyncpg retorna objetos Record, no dicts.
    # Algunos tipos de PostgreSQL no son serializables a JSON directamente.
    sample = []
    for row in rows:
        row_dict = {}
        for key, value in dict(row).items():
            if hasattr(value, "isoformat"):
                # datetime, date, time -> string ISO 8601
                row_dict[key] = value.isoformat()
            elif hasattr(value, "__str__") and not isinstance(
                value, (str, int, float, bool, type(None))
            ):
                # UUID, Decimal, geometry, etc. -> string
                row_dict[key] = str(value)
            else:
                row_dict[key] = value
        sample.append(row_dict)

    return {
        "table": table_name,
        "sample": sample,
        "count": len(sample),
    }

### Tool 4: `query_database()` — Ejecutar SQL (la más peligrosa)

Esta es la **única tool que ejecuta SQL del usuario**. Por eso tiene múltiples capas de protección. Fíjate también en los **mensajes de error descriptivos** — le dicen a Claude cómo corregir el problema en el siguiente step del agent loop.

> **Ejemplo**: si Claude genera un query con una columna que no existe, el error dice *"Columna no encontrada. Usa get_table_schema para ver las columnas disponibles."* Esto guía a Claude a auto-corregirse.

In [ ]:
async def query_database(sql: str) -> dict:
    """
    Ejecuta un SELECT query contra la base de datos.
    
    Capas de proteccion:
    1. Solo permite SELECT (parsea el SQL con sqlparse)
    2. Agrega LIMIT si no lo tiene (maximo 100 rows)
    3. Timeout de 5 segundos (statement_timeout de PostgreSQL)
    4. El usuario de DB es read-only (proteccion a nivel de PostgreSQL)
    
    Los mensajes de error son descriptivos para que Claude
    pueda auto-corregirse en el siguiente step.
    """
    logger.info(f"  [DB] Query: {sql[:200]}")

    # Capa 1: Verificar que sea un SELECT
    if not is_safe_query(sql):
        return {
            "error": "Solo se permiten queries SELECT. "
            "No se pueden ejecutar INSERT, UPDATE, DELETE, DROP, etc."
        }

    # Capa 2: Agregar LIMIT si no lo tiene
    sql = add_limit(sql, max_rows=100)

    # Capa 3 y 4: Ejecutar con timeout + usuario read-only
    try:
        async with pool.acquire() as conn:
            # statement_timeout limita cuanto puede tardar el query.
            # Si tarda mas de 5 segundos, PostgreSQL lo cancela.
            await conn.execute("SET statement_timeout = '5000'")
            rows = await conn.fetch(sql)

        # Convertir a lista de dicts serializables
        results = []
        for row in rows:
            row_dict = {}
            for key, value in dict(row).items():
                if hasattr(value, "isoformat"):
                    row_dict[key] = value.isoformat()
                elif hasattr(value, "__str__") and not isinstance(
                    value, (str, int, float, bool, type(None))
                ):
                    row_dict[key] = str(value)
                else:
                    row_dict[key] = value
            results.append(row_dict)

        logger.info(f"  [DB] Retorno {len(results)} rows")

        return {
            "sql": sql,
            "rows": results,
            "row_count": len(results),
        }

    # Errores descriptivos: cada uno guia a Claude a la solucion
    except asyncpg.exceptions.QueryCanceledError:
        return {
            "error": "Query cancelado: excedio el timeout de 5 segundos. "
            "Intenta un query mas especifico o con menos datos."
        }
    except asyncpg.exceptions.UndefinedTableError as e:
        return {
            "error": f"Tabla no encontrada: {e}. "
            "Usa list_tables() para ver las tablas disponibles."
        }
    except asyncpg.exceptions.UndefinedColumnError as e:
        return {
            "error": f"Columna no encontrada: {e}. "
            "Usa get_table_schema() para ver las columnas disponibles."
        }
    except asyncpg.exceptions.PostgresSyntaxError as e:
        return {"error": f"Error de sintaxis SQL: {e}"}
    except Exception as e:
        return {"error": f"Error ejecutando query: {type(e).__name__}: {e}"}

---

## 7.3 Integración con PostgreSQL

### asyncpg vs psycopg2

| Característica | `psycopg2` | `asyncpg` |
|---------------|-----------|----------|
| **Modo** | Síncrono | Asíncrono |
| **Velocidad** | Base | ~3x más rápido |
| **Popularidad** | El más usado | Creciendo rápido |
| **Connection pool** | Requiere extensión (`psycopg2.pool`) | Pool nativo |
| **Tipos PostgreSQL** | Buenos | Excelentes (mejor mapeo) |
| **Ideal para** | Scripts simples, Django | Agent loops async, FastAPI |

Para este track usamos **asyncpg** porque:
1. El agent loop ya es async (usamos `AsyncAnthropic`)
2. El pool nativo simplifica la configuración
3. Es más rápido, lo cual importa cuando el agente hace 5-10 queries por pregunta

### Connection Pooling — por qué importa

Sin pool, cada query abre y cierra una conexión. Abrir una conexión a PostgreSQL tarda ~50-100ms. Si tu agente hace 10 queries, son **500-1000ms solo en conexiones**.

Con pool, las conexiones se mantienen abiertas y se reusan. `pool.acquire()` toma una conexión del pool (microsegundos), y al salir del `async with`, la conexión **vuelve al pool** (no se cierra).

```
SIN pool (1000ms en conexiones):
  Query 1: [open 80ms][query 20ms][close]
  Query 2: [open 80ms][query 20ms][close]
  ...
  Query 10: [open 80ms][query 20ms][close]

CON pool (0ms en conexiones despues del setup):
  Query 1: [acquire ~0ms][query 20ms][release ~0ms]
  Query 2: [acquire ~0ms][query 20ms][release ~0ms]
  ...
  Query 10: [acquire ~0ms][query 20ms][release ~0ms]
```

In [ ]:
# SIN pool (malo para agentes -- 50-100ms por conexion):
async def query_bad(sql):
    conn = await asyncpg.connect(
        host="localhost", database="klimbook",
        user="readonly_agent", password="secure_password",
    )  # 50-100ms cada vez
    result = await conn.fetch(sql)
    await conn.close()
    return result


# CON pool (bueno para agentes -- conexiones se reusan):

# El pool se crea UNA vez al inicio de la aplicacion
pool = await asyncpg.create_pool(
    host="localhost",
    database="klimbook",
    user="readonly_agent",
    password="secure_password",
    min_size=2,   # siempre tener 2 conexiones listas
    max_size=10,  # nunca tener mas de 10
)


async def query_good(sql):
    # acquire() toma una conexion del pool (microsegundos).
    # Al salir del 'async with', la conexion vuelve al pool (no se cierra).
    async with pool.acquire() as conn:
        return await conn.fetch(sql)

### Query Timeout — protección contra queries lentos

Un agente podría generar un query que tarde minutos (ej: un JOIN cartesiano entre dos tablas grandes). Necesitas un timeout para cancelar queries que tarden demasiado.

Hay tres niveles donde puedes configurar timeout:

In [ ]:
import asyncio

# Opcion 1: A nivel de conexion (aplica a todos los queries en esa conexion).
# Es lo que usamos en query_database().
# SET es un comando de PostgreSQL que configura la sesion actual.
async def query_with_pg_timeout(conn, sql):
    await conn.execute("SET statement_timeout = '5000'")  # 5 segundos
    return await conn.fetch(sql)


# Opcion 2: A nivel de pool (aplica a TODAS las conexiones).
# command_timeout es un parametro de asyncpg, no de PostgreSQL.
pool_with_timeout = await asyncpg.create_pool(
    host="localhost",
    database="klimbook",
    command_timeout=10,  # 10 segundos para cualquier query
)


# Opcion 3: A nivel de query individual con asyncio.
# Util si necesitas timeouts diferentes para queries diferentes.
async def query_with_asyncio_timeout(conn, sql, timeout=5.0):
    try:
        result = await asyncio.wait_for(
            conn.fetch(sql),
            timeout=timeout,
        )
        return result
    except asyncio.TimeoutError:
        return {"error": "Query excedio el timeout"}

### Formateo de resultados: PostgreSQL → JSON

`asyncpg` retorna objetos `Record` que no son directamente serializables a JSON. Además, PostgreSQL tiene tipos de datos que JSON no soporta nativamente:

| Tipo PostgreSQL | Problema | Conversión |
|----------------|---------|-----------|
| `datetime` / `date` / `time` | No existe en JSON | `.isoformat()` → string ISO 8601 |
| `Decimal` | JSON solo tiene `float` | `float()` (pierde precisión pero es serializable) |
| `UUID` | No existe en JSON | `str()` |
| `bytes` / `bytea` | Datos binarios | `"<binary data>"` o base64 |
| `geometry` (PostGIS) | Tipo custom | `str()` → WKT, o extraer lat/lon |

In [ ]:
def row_to_dict(row) -> dict:
    """
    Convierte un asyncpg Record a un dict serializable a JSON.
    
    Maneja los tipos especiales de PostgreSQL que JSON no soporta.
    Esta funcion se usa en get_sample_data() y query_database().
    """
    result = {}
    for key, value in dict(row).items():
        if value is None:
            result[key] = None
        elif isinstance(value, (str, int, float, bool)):
            # Tipos nativos de JSON -- pasan directo
            result[key] = value
        elif hasattr(value, "isoformat"):
            # datetime, date, time -> ISO 8601 string
            result[key] = value.isoformat()
        elif isinstance(value, bytes):
            # Datos binarios -- no serializables, skip o base64
            result[key] = "<binary data>"
        else:
            # Fallback: UUID, Decimal, geometry, etc.
            result[key] = str(value)
    return result


# Ejemplo de uso (simulado):
from datetime import datetime
from decimal import Decimal

class FakeRecord:
    """Simula un asyncpg Record para demostrar la conversion."""
    def __init__(self, data):
        self._data = data
    def __iter__(self):
        return iter(self._data.items())

# Simular un row con tipos problematicos
fake_row_data = {
    "id": 42,
    "username": "alex_climber",
    "created_at": datetime(2026, 3, 15, 14, 30, 0),
    "balance": Decimal("19.99"),
    "avatar": b"\x89PNG\r\n",
    "is_active": True,
    "email": None,
}

# Convertir manualmente para demostrar
converted = {}
for key, value in fake_row_data.items():
    if value is None:
        converted[key] = None
    elif isinstance(value, (str, int, float, bool)):
        converted[key] = value
    elif hasattr(value, "isoformat"):
        converted[key] = value.isoformat()
    elif isinstance(value, bytes):
        converted[key] = "<binary data>"
    else:
        converted[key] = str(value)

print("Resultado convertido:")
for k, v in converted.items():
    print(f"  {k}: {v!r} ({type(v).__name__})")

---

## 7.4 PostGIS: Queries Geoespaciales

La base de datos de Klimbook usa **PostGIS** para datos geográficos (ubicación de crags, rutas, etc.). Claude puede generar queries espaciales si le das el contexto correcto.

### ¿Qué es PostGIS?

Es una **extensión de PostgreSQL** que agrega tipos de datos geográficos y funciones para trabajar con ellos. En vez de guardar latitud y longitud como dos floats separados, guardas un **punto geométrico** que PostgreSQL puede indexar y consultar eficientemente.

| Aspecto | Sin PostGIS | Con PostGIS |
|---------|-------------|-------------|
| **Almacenamiento** | `lat FLOAT, lon FLOAT` | `geom GEOMETRY(Point, 4326)` |
| **Búsqueda por área** | `WHERE lat BETWEEN x AND y AND lon BETWEEN a AND b` (rectángulo, inexacto) | `ST_DWithin(geom, punto, radio)` (círculo, exacto) |
| **Índices** | No puede usar índices espaciales | Usa GiST index, muy rápido |
| **Distancia** | Fórmula de Haversine manual | `ST_Distance(geom1, geom2)` en metros |

In [ ]:
# Sin PostGIS (malo -- busca en un RECTANGULO, no en un circulo):
query_sin_postgis = """
SELECT * FROM crags 
WHERE latitude BETWEEN 18 AND 20 
AND longitude BETWEEN -99 AND -97;
"""
# No es preciso (un rectangulo no es un circulo).
# No puede usar indices espaciales eficientes.


# Con PostGIS (bueno -- busca en un CIRCULO con indice espacial):
query_con_postgis = """
SELECT 
    name, 
    ST_Distance(
        geom::geography, 
        ST_MakePoint(-98.20, 19.04)::geography
    ) / 1000 AS distance_km
FROM crags
WHERE ST_DWithin(
    geom::geography, 
    ST_MakePoint(-98.20, 19.04)::geography, 
    50000  -- 50km en metros
)
ORDER BY distance_km;
"""
# Busca en un circulo de 50km, usa indices, y es preciso.
# ::geography convierte a coordenadas esfericas (distancia en metros).

print("Sin PostGIS:", query_sin_postgis.strip()[:60], "...")
print("Con PostGIS:", query_con_postgis.strip()[:60], "...")

### Funciones PostGIS que Claude debe conocer

Para que Claude genere queries PostGIS correctos, necesitas incluir esta referencia en el system prompt o en la `description` de la tool `query_database`. Sin esto, Claude va a inventar nombres de funciones o usar la sintaxis incorrecta.

| Función | Qué hace | Ejemplo |
|---------|----------|---------|
| `ST_Distance(g1, g2)` | Distancia entre dos geometrías. Con `::geography` retorna **metros** | `ST_Distance(geom::geography, ST_MakePoint(-98.2, 19.0)::geography)` |
| `ST_DWithin(g1, g2, d)` | True si distancia < d metros. **Usa índices**, mucho más rápido que filtrar con ST_Distance | `WHERE ST_DWithin(geom::geography, punto::geography, 50000)` |
| `ST_MakePoint(lon, lat)` | Crea un punto. **ORDEN: (longitud, latitud)**, NO (latitud, longitud) | `ST_MakePoint(-98.20, 19.04)` = Puebla |
| `ST_AsText(geom)` | Convierte geometría a texto legible (WKT) | `ST_AsText(geom)` → `'POINT(-98.20 19.04)'` |
| `ST_X(geom)` / `ST_Y(geom)` | Extraen longitud / latitud de un punto | `ST_X(geom)` = longitud, `ST_Y(geom)` = latitud |

> **Error muy común**: `ST_MakePoint` recibe **(longitud, latitud)**, no (latitud, longitud). Si Claude lo pone al revés, el query retorna resultados en el **otro lado del mundo**. Asegúrate de incluir esto explícitamente en el contexto.

### `::geography` vs `::geometry`

| Cast | Coordenadas | Distancia en | Cuándo usar |
|------|------------|-------------|-------------|
| `::geometry` | Planas (grados) | Grados (inútil) | Nunca para distancias reales |
| `::geography` | Esféricas (tierra real) | **Metros** | Siempre para distancias en el mundo real |

**Regla**: siempre usar `::geography` cuando calcules distancias. La opción `::geometry` mide en grados, lo cual no tiene sentido para "¿cuántos km hay de aquí a allá?".

In [ ]:
# Contexto PostGIS para incluir en el system prompt del agente.
# Esto es lo que Claude necesita para generar queries correctos.

POSTGIS_CONTEXT = """
PostGIS functions available:

ST_Distance(geom1, geom2): 
    Distancia entre dos geometrias. Con ::geography retorna metros.
    Ejemplo: ST_Distance(geom::geography, ST_MakePoint(-98.20, 19.04)::geography)

ST_DWithin(geom1, geom2, distance_meters): 
    True si la distancia es menor a N metros. USA INDICES, mucho mas rapido.
    Ejemplo: ST_DWithin(geom::geography, ST_MakePoint(lon, lat)::geography, 50000)

ST_MakePoint(longitude, latitude): 
    Crea un punto. IMPORTANTE: el orden es (LONGITUD, LATITUD), NO (latitud, longitud).
    Ejemplo: ST_MakePoint(-98.20, 19.04)  -- Puebla

ST_AsText(geom): 
    Convierte geometria a texto legible (WKT).
    Ejemplo: ST_AsText(geom) -> 'POINT(-98.20 19.04)'

ST_X(geom), ST_Y(geom): 
    Extraen longitud y latitud de un punto.
    ST_X = longitud, ST_Y = latitud

SRID 4326: 
    Sistema de coordenadas WGS84 (el que usa GPS).
    Todas las geometrias en esta DB usan SRID 4326.

::geography vs ::geometry:
    - geometry: coordenadas planas, distancia en grados (INUTIL)
    - geography: coordenadas esfericas, distancia en METROS
    Siempre usar ::geography para distancias en el mundo real.
"""

print(POSTGIS_CONTEXT)

### Ejemplos de queries PostGIS que el agente debería poder generar

Estos son los tipos de preguntas geoespaciales que un usuario de Klimbook haría. El agente necesita generar el SQL correcto para cada una:

In [ ]:
# Pregunta: "Que crags hay a menos de 100km de Puebla?"
query_crags_nearby = """
SELECT 
    name,
    ST_Distance(
        geom::geography, 
        ST_MakePoint(-98.2063, 19.0414)::geography
    ) / 1000 AS distance_km
FROM crags
WHERE ST_DWithin(
    geom::geography, 
    ST_MakePoint(-98.2063, 19.0414)::geography, 
    100000  -- 100km en metros
)
ORDER BY distance_km;
"""


# Pregunta: "Cuantas rutas hay en un radio de 50km de mi ubicacion?"
query_routes_near_me = """
SELECT COUNT(*) as total_routes
FROM routes r
JOIN crags c ON r.crag_id = c.id
WHERE ST_DWithin(
    c.geom::geography,
    ST_MakePoint(-98.2063, 19.0414)::geography,
    50000  -- 50km en metros
);
"""


# Pregunta: "Donde se concentra la actividad de escalada en Mexico?"
# (Agrupar por cuadriculas de ~100km para ver zonas de concentracion)
query_activity_heatmap = """
SELECT 
    ST_AsText(ST_Centroid(ST_Collect(c.geom))) as center,
    COUNT(a.id) as total_ascents,
    COUNT(DISTINCT c.id) as total_crags
FROM ascents a
JOIN routes r ON a.route_id = r.id
JOIN crags c ON r.crag_id = c.id
GROUP BY ST_SnapToGrid(c.geom, 1)  -- cuadricula de ~111km (1 grado)
HAVING COUNT(a.id) > 10
ORDER BY total_ascents DESC;
"""

print("Query 1 (crags cerca):", query_crags_nearby.strip()[:80], "...")
print("Query 2 (rutas cerca):", query_routes_near_me.strip()[:80], "...")
print("Query 3 (heatmap):", query_activity_heatmap.strip()[:80], "...")

---

## 7.5 El Query Validation Pipeline

Antes de ejecutar cualquier SQL generado por Claude, pásalo por este pipeline de validación. Cada paso es una **capa de defensa** — si una falla, las siguientes la respaldan.

```
SQL generado por Claude
    |
    v
1. Parsear con sqlparse
    -> Es un SELECT valido?
    -> Tiene keywords peligrosos?
    |
    v
2. Verificar tablas (opcional pero recomendado)
    -> Las tablas mencionadas existen?
    |
    v
3. Agregar LIMIT
    -> Si no tiene LIMIT, agregar LIMIT 100
    |
    v
4. Configurar timeout
    -> SET statement_timeout = '5000'
    |
    v
5. Ejecutar con usuario read-only
    -> El usuario de DB solo tiene permisos SELECT
    |
    v
6. Formatear resultados
    -> Convertir tipos PostgreSQL a JSON
    -> Truncar si hay demasiados datos
```

El paso 2 (verificar tablas) es **opcional** pero añade una capa extra: si Claude inventa un nombre de tabla, el error es más descriptivo que el genérico de PostgreSQL y puedes incluir la lista de tablas disponibles para que Claude se auto-corrija.

### Implementación completa del pipeline

Esta función unifica todos los pasos de validación en una sola llamada. Es lo que usarías en producción en vez de `query_database()` directamente:

In [ ]:
import re


def extract_table_names(sql: str) -> list[str]:
    """
    Extrae nombres de tablas mencionados en un query SQL.
    
    Busca la palabra despues de FROM y JOIN.
    No es perfecto (no maneja subconsultas complejas),
    pero cubre el 90% de los casos.
    """
    # Patron: FROM tabla o JOIN tabla (con alias opcional)
    pattern = r"(?:FROM|JOIN)\s+(\w+)"
    matches = re.findall(pattern, sql, re.IGNORECASE)
    # Filtrar palabras clave que no son tablas
    keywords = {"select", "where", "on", "and", "or", "not", "in", "as"}
    return [m for m in matches if m.lower() not in keywords]


async def safe_query(sql: str) -> dict:
    """
    Pipeline completo de validacion y ejecucion de SQL.
    
    Combina todas las capas de seguridad:
    1. Verificar que sea SELECT (sqlparse)
    2. Verificar que las tablas existan (opcional)
    3. Agregar LIMIT si no lo tiene
    4. Ejecutar con timeout + usuario read-only
    5. Formatear resultados a JSON
    """
    # Paso 1: Verificar que sea SELECT
    if not is_safe_query(sql):
        return {"error": "Solo se permiten queries SELECT."}

    # Paso 2: Verificar que las tablas existan.
    # Si una tabla no existe, retornamos error descriptivo con la lista
    # de tablas disponibles para que Claude se auto-corrija.
    tables_in_query = extract_table_names(sql)
    existing = await list_tables()
    existing_tables = existing.get("tables", [])

    for table in tables_in_query:
        if table not in existing_tables:
            return {
                "error": f"Tabla '{table}' no existe.",
                "available_tables": existing_tables,
            }

    # Paso 3: Agregar LIMIT si no lo tiene
    sql = add_limit(sql, max_rows=100)

    # Paso 4: Ejecutar con timeout + usuario read-only
    try:
        async with pool.acquire() as conn:
            await conn.execute("SET statement_timeout = '5000'")
            rows = await conn.fetch(sql)
    except Exception as e:
        return {"error": str(e)}

    # Paso 5: Formatear resultados
    results = [row_to_dict(row) for row in rows]

    return {
        "sql": sql,
        "rows": results,
        "row_count": len(results),
    }


# Probar extract_table_names
test_queries = [
    "SELECT * FROM users",
    "SELECT u.name, r.grade FROM users u JOIN routes r ON u.id = r.user_id",
    "SELECT * FROM ascents a JOIN routes r ON a.route_id = r.id JOIN crags c ON r.crag_id = c.id",
]

for q in test_queries:
    tables = extract_table_names(q)
    print(f"  Query: {q[:60]}...")
    print(f"  Tablas: {tables}\n")

### Limitaciones del pipeline de validación

El pipeline no es perfecto. Hay ataques más sofisticados que podrían pasar:

| Ataque | Ejemplo | ¿Lo detecta? | Mitigación |
|--------|---------|-------------|-----------|
| DROP TABLE | `DROP TABLE users` | Si — `is_safe_query` lo rechaza | Parseo sqlparse |
| DELETE dentro de SELECT | `SELECT * FROM (DELETE FROM users RETURNING *) sub` | Si — detecta keyword `DELETE` | Búsqueda de keywords |
| Funciones peligrosas | `SELECT pg_read_file('/etc/passwd')` | **No** — es un SELECT válido | Usuario read-only no tiene acceso a `pg_read_file` |
| Timing attacks | `SELECT pg_sleep(300)` | Parcialmente — timeout lo cancela después de 5s | `statement_timeout` |
| Information leaking | `SELECT * FROM pg_shadow` (hashes de passwords) | **No** — es un SELECT válido | Usuario read-only no tiene acceso a tablas del sistema |

Por eso la **capa más importante es el usuario read-only de PostgreSQL**. Incluso si el parseo falla, los permisos de la DB son la última barrera. Nunca uses un usuario con permisos de escritura para el agente.

> **Principio de mínimo privilegio**: dale al agente los permisos mínimos que necesita para funcionar. Si solo necesita leer datos, el usuario de DB solo debería poder hacer SELECT.

---

## Resumen de Conceptos Clave — Semana 7

### Arquitectura del agente de base de datos

```
Usuario: pregunta en lenguaje natural
    |
    v
Agent Loop (Claude + tools)
    |
    +-- list_tables()           -> que tablas hay
    +-- get_table_schema(t)     -> columnas, tipos, FKs
    +-- get_sample_data(t, n)   -> datos reales de ejemplo
    +-- query_database(sql)     -> ejecutar SELECT con protecciones
    |
    v
Claude: interpreta resultados y responde
```

### Conceptos principales

| Concepto | Descripción |
|----------|-------------|
| **Text-to-SQL** | Claude genera SQL a partir de lenguaje natural. Necesita conocer el schema para no inventar tablas/columnas |
| **4 tools para DB** | `list_tables`, `get_table_schema`, `get_sample_data`, `query_database`. El agente explora el schema antes de hacer queries |
| **Defensa en profundidad** | Solo SELECT + usuario read-only + LIMIT + timeout + validación sqlparse. Nunca confíes en una sola capa |
| **Connection pooling** | Crear el pool una vez con `asyncpg.create_pool()`, reutilizar conexiones. ~50-100ms ahorrados por query |
| **PostGIS** | `ST_DWithin` para buscar por proximidad, `ST_Distance` para distancias. Orden: **(longitud, latitud)** |
| **`::geography`** | Siempre usar `::geography` para distancias reales en metros. `::geometry` mide en grados (inútil) |
| **Errores descriptivos** | Los mensajes de error de las tools guían a Claude a auto-corregirse (ej: "Usa get_table_schema para ver columnas") |

### Checklist de seguridad para Text-to-SQL

- [ ] Solo ejecutar `SELECT` — parsear con `sqlparse` y rechazar todo lo demás
- [ ] Usuario de DB **read-only** — `GRANT SELECT` únicamente
- [ ] `LIMIT` obligatorio — agregar automáticamente si el query no lo tiene
- [ ] `statement_timeout` — cancelar queries que tarden más de N segundos
- [ ] Validar nombres de tabla — solo alfanuméricos y `_` (prevenir SQL injection)
- [ ] Formatear resultados — convertir tipos PostgreSQL a JSON serializable
- [ ] Mensajes de error descriptivos — guiar al agente a auto-corregirse

### Modelo mental: cuándo usar cada tool

```
Claude recibe la pregunta del usuario
    |
    "No se que tablas hay"
    -> list_tables()
    |
    "No conozco las columnas de esta tabla"
    -> get_table_schema(tabla)
    |
    "Necesito ver los datos reales para entender el formato"
    -> get_sample_data(tabla, 5)
    |
    "Ya tengo toda la info, puedo escribir el SQL"
    -> query_database(sql)
    |
    "El query fallo, necesito corregirlo"
    -> get_table_schema(tabla) otra vez, luego query_database(sql_corregido)
```